In [183]:
import matplotlib.pyplot as plt
import numpy as np
import scipy
import os, cv2
from scipy.signal import convolve
import scipy
from scipy import optimize
%matplotlib

Using matplotlib backend: QtAgg


In [109]:
image = cv2.imread("C:\\Users\\RND\\EFI-200\\SFR_calculation_code\\slanted_2_crp_flipped.tiff", cv2.IMREAD_GRAYSCALE)
image.dtype

dtype('uint8')

In [110]:
1000 / 2.74 / 2



182.4817518248175

In [111]:
import sys
sys.path.append(r'c:\Users\RND\EFI-200\SFR_calculation_code')

In [112]:
folder = "C:\\Users\\RND\\EFI-200\\SFR_calculation_code\\"
im_filename = "slanted_2_crp.tiff" 
full_im_path = os.path.join(folder, im_filename)
import utils
folder_basename = os.path.basename(folder)
print(f'Processing {os.path.join(folder_basename, im_filename)}')
im_roi = cv2.imread(full_im_path, cv2.IMREAD_GRAYSCALE)  # read image data

if len(im_roi.shape) > 2:
    im_roi = im_roi[:, :, 0]  # TODO find a more general way to handle RGB and RGBA images ************************
roi_height, roi_width = im_roi.shape

Processing slanted_2_crp.tiff


In [113]:
dpi=200
lam_diffr=550e-9
f_number = 3.8
pixel_pitch = 2.4 #um

# Prepare MTF curve plot object
mtf_plotter = utils.MTFplotter(pixel_pitch, f_number, lam_diffr=lam_diffr, fit_begin=[0.54, 0.63],
                                fit_end=[0.90, 0.81], mtf_fit_limit=[0.40], mtf_tail_lvl=0.05)

In [114]:
for x_lims, range_str in zip([(0, 250), (0, 400)], ['', '_0to400cymm']):
    fig, [ax1, ax2] = plt.subplots(1, 2, figsize=(12, 4))

    ax1.imshow(im_roi, vmin=0.0, vmax=1.0, cmap='gray')

    # Plot MTF curves and return MTF data
    mtf_system, mtf_lens, status = mtf_plotter.calc_and_plot_mtf(ax2, im_roi, x_lims=x_lims)

    angle = status['angle']  # edge angle relative to vertical

    file_basename = "C:\\Users\\RND\\EFI-200\\SFR_calculation_code\\slanted_2_crp.tiff"
    file_image_save_name = file_basename + f'_mtf{range_str}.png'
    plt.savefig(file_image_save_name, dpi=dpi)

file_data_save_name = file_basename + f'_mtf_system.txt'
np.savetxt(file_data_save_name, mtf_system, fmt='%.4e')

file_data_save_name = file_basename + f'_mtf_lens.txt'
np.savetxt(file_data_save_name, mtf_lens, fmt='%.4e')

# Obtain lens MTF at the design specification frequencies
specification_freqs = np.linspace(0, 400, 2000)
mtf_lens_interp = scipy.interpolate.interp1d(mtf_lens[:, 0], mtf_lens[:, 1])
for spatial_freq in specification_freqs:
    mtf_at_sf = mtf_lens_interp(spatial_freq)[()]
    print(f'MTF at {spatial_freq:.0f} cyc/mm is {mtf_at_sf * 100:.0f}%') 

MTF at 0 cyc/mm is 100%
MTF at 0 cyc/mm is 100%
MTF at 0 cyc/mm is 100%
MTF at 1 cyc/mm is 100%
MTF at 1 cyc/mm is 100%
MTF at 1 cyc/mm is 100%
MTF at 1 cyc/mm is 100%
MTF at 1 cyc/mm is 99%
MTF at 2 cyc/mm is 99%
MTF at 2 cyc/mm is 99%
MTF at 2 cyc/mm is 99%
MTF at 2 cyc/mm is 99%
MTF at 2 cyc/mm is 99%
MTF at 3 cyc/mm is 99%
MTF at 3 cyc/mm is 99%
MTF at 3 cyc/mm is 99%
MTF at 3 cyc/mm is 99%
MTF at 3 cyc/mm is 99%
MTF at 4 cyc/mm is 99%
MTF at 4 cyc/mm is 99%
MTF at 4 cyc/mm is 99%
MTF at 4 cyc/mm is 98%
MTF at 4 cyc/mm is 98%
MTF at 5 cyc/mm is 98%
MTF at 5 cyc/mm is 98%
MTF at 5 cyc/mm is 98%
MTF at 5 cyc/mm is 98%
MTF at 5 cyc/mm is 98%
MTF at 6 cyc/mm is 98%
MTF at 6 cyc/mm is 98%
MTF at 6 cyc/mm is 98%
MTF at 6 cyc/mm is 98%
MTF at 6 cyc/mm is 98%
MTF at 7 cyc/mm is 98%
MTF at 7 cyc/mm is 98%
MTF at 7 cyc/mm is 97%
MTF at 7 cyc/mm is 97%
MTF at 7 cyc/mm is 97%
MTF at 8 cyc/mm is 97%
MTF at 8 cyc/mm is 97%
MTF at 8 cyc/mm is 97%
MTF at 8 cyc/mm is 97%
MTF at 8 cyc/mm is 97%
MTF 

In [195]:
def dot(a, b):
    return a[0] * b[0] + a[1] * b[1]

def angle_from_slope(slope):
    return np.rad2deg(np.arctan(slope))

def slope_from_angle(angle):
    return np.tan(np.deg2rad(angle))

def cubic_solver(a, b, c, d):
    # Solve the equation a*x**3 + b*x**2 + c*x + d = 0 for a
    # real-valued root x by Cardano's method
    # (https://en.wikipedia.org/wiki/Cubic_equation#Cardano's_formula)

    p = (3 * a * c - b ** 2) / (3 * a ** 2)
    q = (2 * b ** 3 - 9 * a * b * c + 27 * a ** 2 * d) / (27 * a ** 3)

    # A real root exists if 4 * p**3 + 27 * q**2 > 0
    sr = np.sqrt(q ** 2 / 4 + p ** 3 / 27)
    t = np.cbrt(-q / 2 + sr) + np.cbrt(-q / 2 - sr)
    x = t - b / (3 * a)
    return x

def peak_width(y, rel_threshold):
    # Find width of peak in y that is above a certain fraction of the maximum value
    val = np.abs(y)
    val_threshold = rel_threshold * np.max(val)
    indices = np.where(val - val_threshold > 0.0)[0]
    return indices[-1] - indices[0]

def centroid(arr, conv_kernel = 5, win_width = 5):
    height, width = arr.shape
    win = np.zeros(arr.shape)
    for i in range(height):
        win_c = np.argmax(np.abs(np.convolve(arr[i, :], np.ones(conv_kernel), 'same')))
        win[i, win_c - win_width:win_c + win_width] = 1.0
    x, _ = np.meshgrid(np.arange(width), np.arange(height))
    sum_arr = np.sum(arr * win, axis=1)
    sum_arr_x = np.sum(arr * win * x, axis=1)
    # By design, the following division will result in nan for any row that lack an
    # edge transition
    with np.errstate(divide='ignore', invalid='ignore'):
        return sum_arr_x / sum_arr  # suppress divide-by-zero warnings

def differentiate(arr, diff_kernel = np.array([0.5, 0.0, -0.5])):
    if len(arr.shape) == 2:
        # Use 2-d convolution, but with a one-dimensional (row-oriented) kernel
        out = scipy.signal.convolve2d(arr, [diff_kernel], 'same', 'symm')
    else:
        # Input is a one-dimensional array
        out = np.convolve(arr, diff_kernel, 'same')
        # The first element is not valid since there is no 'symm' option,
        # replace it with 0.0 (thereby maintaining the input array size)
        out[0] = 0.0
    return out

def find_edge(centr, patch_shape, rotated, verbose = False):
        # Find 2nd and 1st order polynomials that best approximate the
        # edge shape given by the vector of LSF centroids supplied  in "centr"
        #
        # input
        #   centr: centroid location for each row
        #   patch_shape: tuple with (height, width) info about the patch
        # output
        #   pcoefs: 2nd order polynomial coefs from the least squares fit to the edge
        #   [slope, offset]: polynomial coefs from the linear fit to the edge

        # Weed out positions in the vector of centroid values that
        # contain nan or inf. These positions represent rows that lack
        # an edge transition. Remove also the first and last values.
        idx = np.where(np.isfinite(centr))[0][1:-1]

        # Find the location and direction of the edge by fitting a line to the
        # centroids on the form x = y*slope + offset
        slope, offset = np.polyfit(idx, centr[idx], 1)

        # pcoefs contains quadratic polynomial coefficients for the x-coordinate
        # of the curved edge as a function of the y-coordinate:
        # x = pcoefs[0] * y**2 + pcoefs[1] * y + pcoefs[2]
        pcoefs = np.polyfit(idx, centr[idx], 2)

        return pcoefs, slope, offset, idx

def calc_distance(data_shape, p, quadratic_fit = True, verbose = False):
        # Calculate the distance (with sign) from each point (x, y) in the
        # image patch "data" to the slanted edge described by the polynomial p.
        # It is assumed that the edge is approximately vertically orientated
        # (between -45° and 45° from the vertical direction).
        # Distances to points to the left of the edge are negative, and positive
        # to points to the right of the edge.
        x, y = np.meshgrid(range(data_shape[1]), range(data_shape[0]))

        verbose and print(f'quadratic fit: {str(quadratic_fit):s}')

        if not quadratic_fit or p[0] == 0.0:
            slope, offset = p[1], p[2]  # use linear fit to edge
            a, b, c = 1, -slope, -offset
            a_b = np.sqrt(a ** 2 + b ** 2)

            # |ax+by+c| / |a_b| is the distance from (x,y) to the slanted edge:
            dist = (a * x + b * y + c) / a_b
        else:
            # Define a cubic polynomial equation for the y-coordinate
            # y0 at the point (x0, y0) on the curved edge that is closest to (x, y)
            d = -y + p[1] * p[2] - x * p[1]
            c = 1 + p[1] ** 2 + 2 * p[2] * p[0] - 2 * x * p[0]
            b = 3 * p[1] * p[0]
            a = 2 * p[0] ** 2

            if p[0] == 0.0:
                y0 = -d / c  # solution if edge is straight (quadratic term is zero)
            else:
                y0 = cubic_solver(a, b, c, d)  # edge is curved

            x0 = p[0] * y0 ** 2 + p[1] * y0 + p[2]
            dxx_dyy = np.array(2 * p[0] * y0 + p[1])  # slope at (x0, y0)
            r2 = dot([1, -dxx_dyy], [1, -dxx_dyy])
            # distance between (x, y) and (x0, y0) along normal to curve at (x0, y0)
            dist = dot([x - x0, y - y0], [1, -dxx_dyy]) / np.sqrt(r2)
        return dist

def project_and_bin(data, dist, oversampling = 4, verbose = False):
        # p contains quadratic polynomial coefficients for the x-coordinate
        # of the curved edge as a function of the y-coordinate:
        # x = p[0]*y**2 + p[1]*y + p[2]

        # Create a matrix "bins" where each element represents the bin index of the
        # corresponding image pixel in "data":
        bins = np.round(dist * oversampling).astype(int)
        bins = bins.flatten()
        bins -= np.min(bins)  # add an offset so that bins start at 0

        esf = np.zeros(np.max(bins) + 1)  # Edge spread function
        cnts = np.zeros(np.max(bins) + 1).astype(int)
        data_flat = data.flatten()
        for b_indx, b_sorted in zip(np.argsort(bins), np.sort(bins)):
            esf[b_sorted] += data_flat[b_indx]  # Collect pixel contributions in this bin
            cnts[b_sorted] += 1  # Keep a tab of how many contributions were made to this bin

        # Calculate mean by dividing by the number of contributing pixels. Avoid
        # division by zero, in case there are bins with no content.
        esf[cnts > 0] /= cnts[cnts > 0]
        if np.any(cnts == 0):
            if verbose:
                print("Warning: esf bins with zero pixel contributions were found. Results may be inaccurate.")
                print(f"Try reducing the oversampling factor, which currently is {oversampling:d}.")
            # Try to save the situation by patching in values in the empty bins if possible
            patch_cntr = 0
            for i in np.where(cnts == 0)[0]:  # loop through all empty bin locations
                j = [i - 1, i + 1]  # indices of nearest neighbors
                if j[0] < 0:  # Is left neighbor index outside esf array?
                    j = j[1]
                elif j[1] == len(cnts):  # Is right neighbor index outside esf array?
                    j = j[0]
                if np.all(cnts[j] > 0):  # Now, if neighbor bins are non-empty
                    esf[i] = np.mean(esf[j])  # use the interpolated value
                    patch_cntr += 1
            if patch_cntr > 0 and verbose:
                print(f"Values in {patch_cntr:d} empty ESF bins were patched by "
                      f"interpolation between their respective nearest neighbors.")
        return esf

def filter_window(lsf, oversampling = 4, lsf_centering_kernel_sz = 9, win_width_factor = 1.5, lsf_threshold = 0.10):
        # The window ('hann_win') returned by this function will be used as a filter
        # on the LSF signal during the MTF calculation to reduce noise
        #Gaussian peak fitting functions
        fitfunc = lambda p, x: p[0]*np.exp(-(x-p[1])**2/(2*p[2]**2)) 
        errfunc = lambda p, x, y: fitfunc(p, x) - y
        x =  np.arange(len(lsf))
        y = lsf
        p0 = [15.5, 2011, 45]
        [p1, success] = optimize.leastsq(errfunc, p0[:], args=(x,y))
        nn0 = 20 * oversampling  # sample range to be used for the FFT, intial guess
        mid = int(p1[1])
        i1 = max(0, mid - nn0)
        i2 = min(2 * mid, mid + nn0)
        nn = (i2 - i1) // 2  # sample range to be used, final

        # Filter LSF curve with a uniform kernel to better find center and
        # determine an appropriate Hann window width for noise reduction
        lsf_conv = np.convolve(lsf[i1:i2], np.ones(lsf_centering_kernel_sz), 'same')

        # Base Hann window half width on the width of the filtered LSF curve
        hann_hw = max(np.round(win_width_factor * peak_width(lsf_conv, lsf_threshold)).astype(int),
                      5 * oversampling)

        bin_c =  np.argmax(np.abs(lsf_conv)) # center bin, corresponding to LSF max

        # Construct Hann window centered over the LSF peak, crop if necessary to
        # the range [0, 2*nn]
        crop_l = max(hann_hw - bin_c, 0)
        crop_r = min(2 * nn - (hann_hw + bin_c), 0)
        hann_win = np.zeros(2 * nn)  # default value outside Hann function
        hann_win[bin_c - hann_hw + crop_l:bin_c + hann_hw + crop_r] = \
            np.hanning(2 * hann_hw)[crop_l:2 * hann_hw + crop_r]
        return hann_win, 2 * hann_hw, [i1, i2]


def calc_mtf(lsf, hann_win, idx, diff_ft = 2, oversampling = 4):
        # Calculate MTF using the LSF as input and use the supplied window function
        # as filter to remove high frequency noise originating in regions far from
        # the edge

        i1, i2 = idx
        mtf = np.abs(np.fft.fft(lsf[i1:i2] * hann_win))
        nn = (i2 - i1) // 2
        mtf = mtf[:nn]
        mtf /= mtf[0]  # normalize to zero spatial frequency
        f = np.arange(0, oversampling / 2, oversampling / nn / 2)  # spatial frequencies (cy/px)
        # Compensate for finite impulse response of the numerical differentiation
        # step used to derive the LSF from the ESF
        # NB: This compensation function is incorrect in both ISO 12233:2014
        # and ISO 12233:2017, Annex D
        mtf *= (1 / np.sinc(4 * f / (diff_ft * oversampling))).clip(0.0, 10.0)
        return np.column_stack((f, mtf))

In [196]:
diff_offset = 0.0
sample_diff = differentiate(image)
centr = centroid(sample_diff) + diff_offset

# Calculate centroids also for the 90° right rotated image
image_rot90 = image.T[:, ::-1]  # rotate by transposing and mirroring
sample_diff = differentiate(image_rot90)
centr_rot = centroid(sample_diff) + diff_offset
verbose = False
if np.sum(np.isnan(centr_rot)) < np.sum(np.isnan(centr)):
    verbose and print("Rotating image by 90°")
    image, centr = image_rot90, centr_rot
    rotated = True
else:
    rotated = False

pcoefs, slope, offset, edge_idx = find_edge(centr, image.shape, rotated)

quadratic_fit = True

pcoefs = [0.0, slope, offset] if not quadratic_fit else pcoefs

dist = calc_distance(image.shape, pcoefs)
esf = project_and_bin(image, dist)  # edge spread function
lsf = differentiate(esf)  # line spread function
hann_win, hann_width, idx = filter_window(lsf)  # define window to be applied on LSF
mtf = calc_mtf(lsf, hann_win, idx)

In [200]:
# esf = 1D edge spread function
x = np.arange(len(esf))

imin = np.min(esf)
imax = np.max(esf)
i10 = imin + 0.1 * (imax - imin)
i90 = imin + 0.9 * (imax - imin)

x10 = np.interp(i10, esf, x)
x90 = np.interp(i90, esf, x)

rise_10_90 = x90 - x10

print(rise_10_90)

47.31078702517493


In [224]:
fig = plt.figure()

#plt.plot(fitted_curve, '-m', label=f"LSF_filtered")
#plt.plot(lsf_conv, '-k', label=f"LSF conv")

#plt.plot(esf, '.k', label=f"ESF, 10-90% rise {rise_10_90:.2f} pixels")
#plt.plot(lsf, '-r', label= "LSF")
#plt.plot(hann_win, '--b', label="Hann window")
plt.plot(mtf[:,0], mtf[:,1], '--g', label="MTF")
plt.legend(loc='best')
plt.axhline(0.2)
plt.grid()
# plt.show()

In [201]:
fig, ax = plt.subplots()
ax.imshow(image, cmap='gray', origin='upper')
ax.plot(centr[edge_idx], edge_idx, '.g', label="centroids")
lin_angle = angle_from_slope(slope)
ax.plot(np.polyval([slope, offset], edge_idx), edge_idx, '-r', label=f"linear fit, tilt angle = {lin_angle:.2f}°")
p_angle = angle_from_slope(pcoefs[1])
ax.plot(np.polyval(pcoefs, edge_idx), edge_idx, '--b', label=f"quadratic fit, tilt angle = {p_angle:.2f}°")
ax.set_xlim([0, image.shape[1]])
ax.set_ylim([image.shape[0], 0])
ax.set_aspect('equal', 'box')
ax.legend(loc='best')
# plt.show()

In [ ]:
""""
Calculate spectral response function (SFR), this is the main function

input: image patch with slanted edge to be analyzed (2-d Numpy array of float)
output: MTF organized as a 2-d array where first column is spatial frequency, and second column
        contains MTF values
output: status dict with fitted edge angle (c.w. from vertical), offset, image rotation etc.

NB: SFR calculations will fail for edge angles of 0°, 45°, and 90° (an inherent limitation of the method)
"""
# TODO: apply Hann (or Hamming) window before calculating centroids, or
# TODO: do a second pass after find_edge with windowing, centroids, and find_edge

# Calculate centroids for the edge transition of each row
diff_offset = 0.0
sample_diff = differentiate(image)

centr = centroid(sample_diff) + diff_offset

# Calculate centroids also for the 90° right rotated image
image_rot90 = image.T[:, ::-1]  # rotate by transposing and mirroring
sample_diff = differentiate(image_rot90)
centr_rot = centroid(sample_diff) + diff_offset

# Use rotated image if it results in fewer rows without edge transitions
verbose = False
if np.sum(np.isnan(centr_rot)) < np.sum(np.isnan(centr)):
    verbose and print("Rotating image by 90°")
    image, centr = image_rot90, centr_rot
    rotated = True
else:
    rotated = False

# Finds polynomials that describes the slanted edge by least squares
# regression to the centroids:
#  - pcoefs are the 2nd order fit coefficients
#  - [slope, offset] are the first order (linear) fit coefficients for the same edge
pcoefs, slope, offset = find_edge(centr, image.shape, rotated)

quadratic_fit = True

pcoefs = [0.0, slope, offset] if not quadratic_fit else pcoefs

# Calculate distance (with sign) from each point (x, y) in the
# image patch "data" to the slanted edge
dist = calc_distance(image.shape, pcoefs)

esf = project_and_bin(image, dist)  # edge spread function

lsf = differentiate(esf)  # line spread function

hann_win, hann_width, idx = filter_window(lsf)  # define window to be applied on LSF

mtf = calc_mtf(lsf, hann_win, idx)